# 10a — Individual direct autoencoder simulators

Train one predictive autoencoder separately for Reid, low-temperature de Pablo, and mixed-temperature de Pablo. Each model uses current displacement, one-frame velocity, and graph structure to predict the next displacement increment through a 2D bottleneck, then feeds predictions back autoregressively. Training remains strictly one-step; architecture, training budget, source-specific splits, rollout cohorts, and metrics match 10b.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT/'src'/'lss').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT/'src') not in sys.path: sys.path.insert(0, str(PROJECT_ROOT/'src'))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lss.latent.direct_autoencoder_simulator import evaluate_direct_autoencoder_rollout, run_direct_autoencoder_case
from lss.latent.experiment import seed_everything
from lss.plotting import PAPER_COLORS, apply_editorial_style, dataset_color, style_axes
from lss.utils import resolve_device
apply_editorial_style()
PAPER_DPI = 180
PAPER_FIGSIZE = (5.4, 4.2)
PAPER_WIDE_FIGSIZE = (10.8, 4.2)
PAPER_TRIPLE_FIGSIZE = (16.2, 4.2)
plt.rcParams.update({'figure.dpi': PAPER_DPI, 'savefig.dpi': 400})
DEVICE = resolve_device('auto')
DEVICE

## Matched configuration

In [ ]:
SEED = 20260811
FORCE_TRAIN = False
TRAIN_TRAJECTORIES = 20
VAL_TRAJECTORIES = 20
TRANSITIONS_PER_TRAJECTORY = 150
# None evaluates every trajectory left after the train/validation split.
EVAL_TRAJECTORIES = None
ROLLOUT_STEPS = [1, 5, 10, 25, 50, 75, 100, 125, 150, 175, 199]
OUTPUT = PROJECT_ROOT/'notebooks'/'results'/'10a_individual_direct_autoencoder_simulators_boxnorm'
OUTPUT.mkdir(parents=True, exist_ok=True)
DATASETS = {
    'reid': {'name': 'reid', 'label': 'Reid', 'path': PROJECT_ROOT/'data'/'reid_200_frames.pt'},
    'depablo_low_temp': {'name': 'depablo_low_temp', 'label': 'de Pablo low-T', 'path': PROJECT_ROOT/'data'/'depablo-near-zero-temp.pt'},
    'depablo_mixed_temp': {'name': 'depablo_mixed_temp', 'label': 'de Pablo mixed-T', 'path': PROJECT_ROOT/'data'/'depablo-10k-mix-temp.pt'},
}
for spec in DATASETS.values(): assert spec['path'].exists(), spec['path']
MODEL_CONFIG = {
    'pos_dim': 2, 'latent_dim': 2, 'latent_tokens': 32, 'hidden_size': 64,
    'autoencoder_model': 'single_stage_attention',
    'node_feature_mode': 'normalized_delta_velocity', 'target_mode': 'normalized_step_delta',
    'edge_mode': 'recomputed_stored', 'batch_graphs': 64, 'mix_sources': False,
    'coordinate_normalization': 'position_normalization',
    'edge_feature_schema': 'physical_static_normalized_edge_changes_v2',
    'learning_rate': 1e-4, 'weight_decay': 1e-5,
    'max_epochs': 15, 'patience': 8, 'min_delta': 1e-5,
}
print({'device': str(DEVICE), 'models': len(DATASETS), 'train trajectories per model': TRAIN_TRAJECTORIES, 'transitions per trajectory': TRANSITIONS_PER_TRAJECTORY})

## Train one model per dataset

Every model sees 20 trajectories and 150 transitions per trajectory. Source-specific split seeds are exactly the seeds used for the corresponding source inside 10b.

In [ ]:
results = {}
for source_index, (case_key, dataset_spec) in enumerate(DATASETS.items()):
    case_seed = SEED + 101*source_index
    case_cfg = {
        **MODEL_CONFIG, 'train_count': TRAIN_TRAJECTORIES, 'val_count': VAL_TRAJECTORIES,
        'transitions_per_trajectory': TRANSITIONS_PER_TRAJECTORY,
        'validation_transitions_per_trajectory': TRANSITIONS_PER_TRAJECTORY,
        'split_seed': case_seed, 'force_train': FORCE_TRAIN,
        'cache_path': str(OUTPUT/f'{case_key}_velocity_residual_direct_autoencoder.pt'),
    }
    print(f"\n{dataset_spec['label']}", flush=True)
    seed_everything(case_seed)
    results[case_key] = run_direct_autoencoder_case(dataset_spec, case_cfg, device=DEVICE)
    display(results[case_key]['split_info'])
    display(results[case_key]['history'].tail())

## One-step validation losses

In [ ]:
fig, ax = plt.subplots(figsize=PAPER_FIGSIZE, constrained_layout=True)
for case_key, result in results.items():
    history = result['history']
    ax.plot(history.epoch, history.val_loss, color=dataset_color(case_key), marker='o', ms=3.5, label=DATASETS[case_key]['label'])
style_axes(ax, xlabel='epoch', ylabel='normalized one-step increment loss', legend=True)
plt.show()

## Fully autoregressive held-out rollouts

In [ ]:
rollout_parts, summary_parts = [], []
for case_key, result in results.items():
    print(f"Rolling out {DATASETS[case_key]['label']}...", flush=True)
    rows, summary = evaluate_direct_autoencoder_rollout(
        result, rollout_steps=ROLLOUT_STEPS, device=DEVICE, max_sims=EVAL_TRAJECTORIES,
    )
    rollout_parts.append(rows); summary_parts.append(summary)
rollout_rows = pd.concat(rollout_parts, ignore_index=True)
rollout_summary = pd.concat(summary_parts, ignore_index=True)
rollout_rows.to_csv(OUTPUT/'heldout_individual_rollout_rows.csv', index=False)
rollout_summary.to_csv(OUTPUT/'heldout_individual_rollout_summary.csv', index=False)
display(rollout_summary.round(5))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=PAPER_WIDE_FIGSIZE, constrained_layout=True)
for case_key, group in rollout_summary.groupby('dataset', sort=False):
    group = group.sort_values('rollout_steps'); color = dataset_color(case_key); label = DATASETS[case_key]['label']
    axes[0].plot(group.rollout_steps, group.p_ratio_r2.clip(0, 1), color=color, marker='o', label=label)
    axes[1].plot(group.rollout_steps, group.position_mse, color=color, marker='o', label=label)
axes[0].axhline(0, color=PAPER_COLORS['ink'], lw=.7, alpha=.5)
style_axes(axes[0], xlabel='rollout step', ylabel='held-out p-ratio R²', legend=True)
style_axes(axes[1], xlabel='rollout step', ylabel='mean node-position MSE', legend=True)
axes[0].set(xlim=(0, max(ROLLOUT_STEPS)), ylim=(0, 1))
axes[1].set_xlim(0, max(ROLLOUT_STEPS)); axes[1].set_yscale('log')
for ax in axes: ax.set_xticks([0, 50, 100, 150, 199])
plt.show()

## Rollout calibration at frame 150

In [ ]:
CALIBRATION_STEP = 150
fig, axes = plt.subplots(1, len(DATASETS), figsize=PAPER_TRIPLE_FIGSIZE, constrained_layout=True, sharex=True, sharey=True)
for ax, (case_key, dataset_spec) in zip(axes, DATASETS.items()):
    group = rollout_rows[rollout_rows.dataset.eq(case_key) & rollout_rows.rollout_steps.eq(CALIBRATION_STEP)].replace([np.inf, -np.inf], np.nan).dropna(subset=['true_p_ratio', 'pred_p_ratio'])
    ax.scatter(group.true_p_ratio, group.pred_p_ratio, s=26, alpha=.72, color=dataset_color(case_key), edgecolor='none')
    if len(group):
        limits=np.r_[group.true_p_ratio,group.pred_p_ratio]; lo,hi=limits.min(),limits.max(); pad=.04*max(hi-lo,1e-6)
        ax.plot([lo-pad,hi+pad],[lo-pad,hi+pad],'--',color=PAPER_COLORS['ink'],lw=1)
    ax.text(.04,.96,dataset_spec['label'],transform=ax.transAxes,ha='left',va='top')
    style_axes(ax,xlabel='true p-ratio',ylabel='predicted p-ratio' if ax is axes[0] else '',legend=False)
plt.show()